# Checks

Run All after every change in `src/netad/`. Stops at the first failing assert.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd

from netad.data import (
    DATA_DIR, FEATURES, CATEGORICAL, NUMERIC, TARGETS, EXPECTED_COLUMNS,
    load_unsw_frame, load_unsw, deduplicate, resolve_conflicts, overlap_with_train,
)

## data.py

### Schema

In [3]:
assert DATA_DIR.is_dir(), DATA_DIR
assert len(CATEGORICAL) == 3
assert len(NUMERIC) == 39
assert len(FEATURES) == 42
assert set(CATEGORICAL).isdisjoint(NUMERIC)
assert len(set(EXPECTED_COLUMNS)) == len(EXPECTED_COLUMNS) == 45

### load_unsw_frame

In [4]:
train_df = load_unsw_frame("train")
test_df = load_unsw_frame("test")

assert train_df.shape == (175341, 45), train_df.shape
assert test_df.shape == (82332, 45), test_df.shape
assert train_df.columns[0] == "id", train_df.columns[0]
assert set(train_df.columns) == set(test_df.columns) == set(EXPECTED_COLUMNS)
assert train_df.isna().sum().sum() == 0
assert test_df.isna().sum().sum() == 0

### load_unsw

In [5]:
X_train, y_train, y_train_multi = load_unsw("train")
X_test, y_test, y_test_multi = load_unsw("test")

assert X_train.shape == (175341, 42), X_train.shape
assert X_test.shape == (82332, 42), X_test.shape
assert list(X_train.columns) == FEATURES
assert not {"id", "label", "attack_cat"} & set(X_train.columns)
assert set(y_train.unique()) == set(y_test.unique()) == {0, 1}
assert ((y_train_multi == "Normal") == (y_train == 0)).all()
assert ((y_test_multi == "Normal") == (y_test == 0)).all()
assert y_train_multi.nunique() == y_test_multi.nunique() == 10
assert all(pd.api.types.is_string_dtype(X_train[c]) for c in CATEGORICAL)
assert all(pd.api.types.is_numeric_dtype(X_train[c]) for c in NUMERIC)

### deduplicate

In [6]:
dedup = deduplicate(train_df)

assert len(dedup) == 107740, len(dedup)
assert not dedup.duplicated(FEATURES + TARGETS).any()
assert dedup.index.isin(train_df.index).all()
assert len(train_df) == 175341

### resolve_conflicts

Row counts after `resolve_conflicts` + `deduplicate`.

In [7]:
expected = {
    ("label", "drop"): 107282,
    ("label", "majority"): 107412,
    ("attack_cat", "drop"): 99268,
    ("attack_cat", "majority"): 100427,
}

for (target, policy), n_rows in expected.items():
    resolved = resolve_conflicts(train_df, target, policy)
    assert resolved.index.isin(train_df.index).all()
    assert (resolved.groupby(FEATURES)[target].nunique() == 1).all(), (target, policy)
    assert len(deduplicate(resolved)) == n_rows, (target, policy, len(deduplicate(resolved)))

In [8]:
assert len(resolve_conflicts(train_df)) == len(resolve_conflicts(train_df, "attack_cat", "majority"))

try:
    resolve_conflicts(train_df, policy="random")
except ValueError:
    pass
else:
    raise AssertionError("bad policy should raise ValueError")

### overlap_with_train

In [9]:
in_train = overlap_with_train(train_df, test_df)

assert in_train.dtype == bool
assert in_train.index.equals(test_df.index)
assert in_train.sum() == 8541, in_train.sum()

### Errors

In [10]:
try:
    load_unsw("validation")
except ValueError:
    pass
else:
    raise AssertionError("bad split should raise ValueError")

try:
    load_unsw("train", data_dir="C:/nu_exista")
except FileNotFoundError:
    pass
else:
    raise AssertionError("missing file should raise FileNotFoundError")

In [11]:
print("all checks passed")

all checks passed
